# Visualizing the entire simulation

In [ ]:
%matplotlib notebook

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import pickle
import pandas as pd
from tqdm import tqdm
import matplotlib as mpl

def visualize_simulation(dynamic_plants=True):
    """
    Visualizes bee movement, plants, and beehives.
    Plant colors are fixed from step 0. Plant positions may change if dynamic_plants=True.
    """

    mpl.rcParams['animation.embed_limit'] = 50

    # Load data
    with open("simulation_data.pkl", "rb") as f:
        df = pickle.load(f)

    bee_df = df[df["Agent Type"] == "bee"]
    plant_df = df[df["Agent Type"] == "plant"]
    beehive_df = df[df["Agent Type"] == "beehive"]
    steps = sorted(bee_df.index.get_level_values("Step").unique())

    # Style from step 0
    plant_style_df = plant_df[plant_df.index.get_level_values("Step") == 0].set_index("unique_id")
    bee_style_df = bee_df[bee_df.index.get_level_values("Step") == 0].set_index("unique_id")
    beehive_df_static = beehive_df[beehive_df.index.get_level_values("Step") == 0]

    # Plot setup
    fig, ax = plt.subplots(figsize=(120/2, 2.85 * 2))
    ax.set_facecolor("gray")

    x_min, x_max = 0, 120
    y_min, y_max = 0, 2.85 * 4
    for x0, y0, x1, y1 in [(x_min, y_min, x_max, y_min), (x_min, y_max, x_max, y_max),
                          (x_min, y_min, x_min, y_max), (x_max, y_min, x_max, y_max)]:
        ax.plot([x0, x1], [y0, y1], color="black", linewidth=1.5)

    all_positions = bee_df["Bee Positions"].tolist() + plant_df["Bee Positions"].tolist() + beehive_df["Bee Positions"].tolist()
    all_x = [p[0] for p in all_positions]
    all_y = [p[1] for p in all_positions]
    ax.set_xlim(min(all_x) - 1, max(all_x) + 1)
    ax.set_ylim(min(all_y) - 1, max(all_y) + 1)
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.set_title("Bee Movement Relative to Plants and Beehives")

    # Static beehives
    for _, hive in beehive_df_static.iterrows():
        pos = hive["Bee Positions"]
        style = hive["Plot Code"]
        ax.scatter(pos[0], pos[1],
                   marker=style.get("marker", "^"),
                   c=style.get("color", "brown"),
                   edgecolors=style.get("edgecolors", "none"),
                   linewidths=style.get("linewidths", 0.8),
                   s=80, zorder=0)
        ax.text(pos[0], pos[1] + 0.5, f"Hive {hive['unique_id']}",
                fontsize=8, ha='center', va='bottom', color='white', zorder=5)

    # Initialize persistent scatter objects
    bee_scatters = {}
    plant_scatters = {}

    for uid, row in plant_style_df.iterrows():
        style = row["Plot Code"]
        scat = ax.scatter([], [],
                          marker=style.get("marker", "o"),
                          c=style.get("color"),
                          edgecolors=style.get("edgecolors", "black"),
                          linewidths=style.get("linewidths", 0.8),
                          s=40, zorder=1)
        plant_scatters[uid] = scat

    for uid, row in bee_style_df.iterrows():
        style = row["Plot Code"]
        scat = ax.scatter([], [],
                          marker=style.get("marker", "o"),
                          c=style.get("color"),
                          edgecolors=style.get("edgecolors", "black"),
                          linewidths=style.get("linewidths", 0.8),
                          s=40, zorder=2)
        bee_scatters[uid] = scat

    # Update logic
    def update(frame):
        current_bees = bee_df.loc[frame]
        if isinstance(current_bees, pd.Series):
            current_bees = pd.DataFrame([current_bees])
        for _, bee in current_bees.iterrows():
            uid = bee["unique_id"]
            if uid in bee_scatters:
                bee_scatters[uid].set_offsets([bee["Bee Positions"]])

        if dynamic_plants and frame in plant_df.index.get_level_values("Step"):
            current_plants = plant_df.loc[frame]
            if isinstance(current_plants, pd.Series):
                current_plants = pd.DataFrame([current_plants])
            for _, plant in current_plants.iterrows():
                uid = plant["unique_id"]
                if uid in plant_scatters:
                    plant_scatters[uid].set_offsets([plant["Bee Positions"]])

        ax.set_title(f"Step {frame} - Bee Movement Relative to Plants and Beehives")
        return list(bee_scatters.values()) + list(plant_scatters.values())

    # Animate
    frame_gen = (i for i in tqdm(steps[::10], desc="Rendering animation"))
    # frame_gen = (i for i in tqdm(steps, desc="Rendering animation"))

    anim = FuncAnimation(fig, update, frames=frame_gen, blit=False, interval=300, repeat=False)

    return anim, HTML(anim.to_jshtml())


In [ ]:
anim, html = visualize_simulation()
html

In [ ]:
with open("bee_animation.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
import webbrowser
webbrowser.open("bee_animation.html")

# Visualisation of specific intervals

In [ ]:
import pickle
import pandas as pd

def load_run_dataframe(pkl_filename, run_id=0):
    """
    Loads a batch results pickle file and extracts the DataFrame for the given RunId.
    
    Parameters:
        pkl_filename (str): Path to the .pkl file.
        run_id (int): RunId to extract. Default = 0.
    
    Returns:
        df_run (pd.DataFrame): DataFrame with MultiIndex (Step, unique_id), filtered to the specified RunId.
    """

    # Load pkl
    with open(pkl_filename, "rb") as f:
        batch_results = pickle.load(f)

    # Convert to DataFrame
    df_all = pd.DataFrame(batch_results)

    # Extract RunID == run_id
    df_run = df_all[df_all["RunId"] == run_id].copy()

    # Set MultiIndex to match expected format
    df_run.set_index(["Step", "unique_id"], inplace=True)

    # Optional diagnostics
    print(f"Loaded RunID {run_id} from '{pkl_filename}' with {len(df_run)} rows")
    print(f"Steps range: {df_run.index.get_level_values('Step').min()} to {df_run.index.get_level_values('Step').max()}")
    print(f"Agent types: {df_run['Agent Type'].unique()}")

    return df_run


In [ ]:
%matplotlib notebook

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
from tqdm import tqdm
import matplotlib as mpl

def visualize_simulation(df, start_time_str="9:00", duration_minutes=10, dynamic_plants=True, frame_skip=1):
    """
    Visualizes bee movement, plants, and beehives within a time interval.
    df: DataFrame of the extracted run, indexed by (Step, unique_id)
    start_time_str: string, e.g. "9:00" (24h format)
    duration_minutes: number of minutes to visualize
    dynamic_plants: whether to update plant positions dynamically
    frame_skip: skip every Nth frame (default=1)
    """

    mpl.rcParams['animation.embed_limit'] = 50

    bee_df = df[df["Agent Type"] == "bee"]
    plant_df = df[df["Agent Type"] == "plant"]
    beehive_df = df[df["Agent Type"] == "beehive"]

    steps = sorted(bee_df.index.get_level_values("Step").unique())

    # --- Time conversion ---
    def time_str_to_seconds(time_str):
        hh, mm = map(int, time_str.split(":"))
        total_seconds = (hh - 9) * 3600 + mm * 60
        return total_seconds

    def step_to_time_str(step):
        total_seconds = step
        hh = 9 + total_seconds // 3600
        mm = (total_seconds % 3600) // 60
        ss = total_seconds % 60
        return f"{hh:02d}:{mm:02d}:{ss:02d}"

    # Compute start and end seconds
    start_second = time_str_to_seconds(start_time_str)
    end_second = start_second + duration_minutes * 60

    # Select steps that fall within [start_second, end_second)
    selected_steps = [step for step in steps if start_second <= step < end_second]

    if len(selected_steps) == 0:
        print("Warning: No steps found in the specified time interval.")
        return None, None

    print(f"Visualizing interval:")
    print(f"  Start time = {step_to_time_str(selected_steps[0])} (Step {selected_steps[0]})")
    print(f"  End time   = {step_to_time_str(selected_steps[-1])} (Step {selected_steps[-1]})")
    print(f"  Total frames to animate (with frame_skip={frame_skip}): {len(selected_steps[::frame_skip])}")

    # --- Style from step 0 ---
    plant_style_df = plant_df.loc[0].reset_index().set_index("unique_id")
    bee_style_df = bee_df.loc[0].reset_index().set_index("unique_id")
    beehive_df_static = beehive_df.loc[0].reset_index()

    # --- Plot setup ---
    fig, ax = plt.subplots(figsize=(120/2, 2.85 * 2))
    ax.set_facecolor("gray")

    x_min, x_max = 0, 120
    y_min, y_max = 0, 2.85 * 4
    for x0, y0, x1, y1 in [(x_min, y_min, x_max, y_min), (x_min, y_max, x_max, y_max),
                          (x_min, y_min, x_min, y_max), (x_max, y_min, x_max, y_max)]:
        ax.plot([x0, x1], [y0, y1], color="black", linewidth=1.5)

    all_positions = bee_df["Bee Positions"].tolist() + plant_df["Bee Positions"].tolist() + beehive_df["Bee Positions"].tolist()
    all_x = [p[0] for p in all_positions]
    all_y = [p[1] for p in all_positions]
    ax.set_xlim(min(all_x) - 1, max(all_x) + 1)
    ax.set_ylim(min(all_y) - 1, max(all_y) + 1)
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.set_title("Bee Movement Relative to Plants and Beehives")

    # --- Static beehives ---
    for _, hive in beehive_df_static.iterrows():
        pos = hive["Bee Positions"]
        style = hive["Plot Code"]
        ax.scatter(pos[0], pos[1],
                   marker=style.get("marker", "^"),
                   c=style.get("color", "brown"),
                   edgecolors=style.get("edgecolors", "none"),
                   linewidths=style.get("linewidths", 0.8),
                   s=80, zorder=0)
        ax.text(pos[0], pos[1] + 0.5, f"Hive {hive['unique_id']}",
                fontsize=8, ha='center', va='bottom', color='white', zorder=5)

    # --- Initialize persistent scatter objects ---
    bee_scatters = {}
    plant_scatters = {}

    for uid, row in plant_style_df.iterrows():
        style = row["Plot Code"]
        scat = ax.scatter([], [],
                          marker=style.get("marker", "o"),
                          c=style.get("color"),
                          edgecolors=style.get("edgecolors", "black"),
                          linewidths=style.get("linewidths", 0.8),
                          s=40, zorder=1)
        plant_scatters[uid] = scat

    for uid, row in bee_style_df.iterrows():
        style = row["Plot Code"]
        scat = ax.scatter([], [],
                          marker=style.get("marker", "o"),
                          c=style.get("color"),
                          edgecolors=style.get("edgecolors", "black"),
                          linewidths=style.get("linewidths", 0.8),
                          s=40, zorder=2)
        bee_scatters[uid] = scat

    # --- Update logic ---
    def update(frame):
        # Bees
        current_bees = bee_df.loc[frame]
        if isinstance(current_bees, pd.Series):
            current_bees = pd.DataFrame([current_bees])
        current_bees = current_bees.reset_index()
        for _, bee in current_bees.iterrows():
            uid = bee["unique_id"]
            if uid in bee_scatters:
                bee_scatters[uid].set_offsets([bee["Bee Positions"]])

        # Plants
        if dynamic_plants and frame in plant_df.index.get_level_values("Step"):
            current_plants = plant_df.loc[frame]
            if isinstance(current_plants, pd.Series):
                current_plants = pd.DataFrame([current_plants])
            current_plants = current_plants.reset_index()
            for _, plant in current_plants.iterrows():
                uid = plant["unique_id"]
                if uid in plant_scatters:
                    plant_scatters[uid].set_offsets([plant["Bee Positions"]])

        ax.set_title(f"Step {frame} ({step_to_time_str(frame)}) - Bee Movement Relative to Plants and Beehives")
        return list(bee_scatters.values()) + list(plant_scatters.values())

    # --- Animate ---
    frame_gen = (i for i in tqdm(selected_steps[::frame_skip], desc="Rendering animation"))

    anim = FuncAnimation(fig, update, frames=frame_gen, blit=False, interval=300, repeat=False)

    return anim, HTML(anim.to_jshtml())


In [ ]:
df_run0 = load_run_dataframe("batch_results_biased_composite.pkl", run_id=0)

anim, html = visualize_simulation(df_run0, start_time_str="11:00", duration_minutes=60)
html

In [ ]:
# Saving the animation to HTML file
with open("simulation_animation.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
# For visualization in a web browser
import webbrowser
webbrowser.open("simulation_animation.html")